# What is A Genetic Algorithm and How do we use it?

Welcome to the fascinating world of Evolutionary Computation. While neural networks mimic the human brain, **Genetic Algorithms (GAs)** mimic biological evolution. They are a subset of machine learning and optimization techniques inspired by Charles Darwin’s theory of natural selection.

GAs are particularly powerful when you are dealing with a **discrete search space** or a **non-differentiable objective function** where standard Gradient Descent algorithms cannot be applied. 

### How does a Genetic Algorithm analytically work?
It operates on a cycle of five core phases:
1. **Initialization:** Create a random population of "individuals" (potential solutions).
2. **Fitness Assignment:** Evaluate how "good" each individual is using a mathematical fitness function.
3. **Selection:** Choose the fittest individuals to be parents.
4. **Crossover (Recombination):** Combine the DNA (parameters) of the parents to create offspring.
5. **Mutation:** Randomly alter some genes in the offspring to maintain genetic diversity and prevent getting stuck in local optima.

In this tutorial, we will implement a GA from scratch using `numpy` to solve the classic **0/1 Knapsack Problem**.

In [ ]:
# Install and import the necessary libraries
# We use numpy for fast array operations and matplotlib for analytical visualization
import numpy as np
import matplotlib.pyplot as plt
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

## Step 1: Defining the Problem and the Fitness Function

The **0/1 Knapsack Problem** asks: Given a set of items, each with a weight and a value, determine which items to include in a collection so that the total weight is less than or equal to a given limit, and the total value is as large as possible.

**Mathematical Formulation:**
Maximize the objective function (Fitness):
$$\text{Fitness} = \sum_{i=1}^{n} v_i x_i$$

Subject to the constraint:
$$\sum_{i=1}^{n} w_i x_i \le W$$

Where:
* $v_i$ is the value of item $i$.
* $w_i$ is the weight of item $i$.
* $x_i \in \{0, 1\}$ represents whether item $i$ is included in the knapsack (our "genes").
* $W$ is the maximum weight capacity.

In [ ]:
# 1. Define the Problem Parameters
NUM_ITEMS = 20
KNAPSACK_CAPACITY = 50

# Generate random weights (between 1 and 10) and values (between 10 and 100)
weights = np.random.randint(1, 10, size=NUM_ITEMS)
values = np.random.randint(10, 100, size=NUM_ITEMS)

print(f"Weights: {weights}")
print(f"Values:  {values}")

# 2. Define the Fitness Function
def calculate_fitness(chromosome):
    """
    Evaluates a single individual.
    A chromosome is a binary array like [1, 0, 1, 1, 0...]
    1 means we take the item, 0 means we leave it.
    """
    total_weight = np.sum(chromosome * weights)
    total_value = np.sum(chromosome * values)
    
    # If the knapsack is over capacity, the individual dies (fitness = 0)
    if total_weight > KNAPSACK_CAPACITY:
        return 0
    else:
        return total_value

# 3. Initialization Function
def initialize_population(pop_size, num_genes):
    """Creates an initial population of random binary arrays."""
    return np.random.randint(2, size=(pop_size, num_genes))

# Test initialization
population = initialize_population(pop_size=5, num_genes=NUM_ITEMS)
print(f"\nSample Chromosome: {population[0]}")
print(f"Sample Fitness: {calculate_fitness(population[0])}")

## Step 2: Selection, Crossover, and Mutation

Once we have a population and can measure their fitness, we need to simulate natural selection.

* **Tournament Selection:** We pick $K$ random individuals from the population and select the one with the highest fitness to become a parent. This is mathematically robust because it maintains selection pressure while preserving some diversity.
* **Single-Point Crossover:** We pick a random "cut" point in the parent chromosomes and swap the tails to create two children.
* **Bit-Flip Mutation:** With a very small probability, we flip a 0 to a 1, or a 1 to a 0. This injects new genetic material into the pool.

In [ ]:
def select_parent(population, fitness_scores, k=3):
    """Tournament selection: Pick k random individuals, return the fittest."""
    # Choose k random indices
    selected_indices = np.random.choice(len(population), size=k, replace=False)
    
    # Find the index of the best individual among the selected
    best_index = selected_indices[np.argmax([fitness_scores[i] for i in selected_indices])]
    return population[best_index]

def crossover(parent1, parent2):
    """Single-point crossover to produce two children."""
    # Pick a random point to split the chromosomes
    crossover_point = random.randint(1, len(parent1) - 1)
    
    # Slice and concatenate
    child1 = np.concatenate([parent1[:crossover_point], parent2[crossover_point:]])
    child2 = np.concatenate([parent2[:crossover_point], parent1[crossover_point:]])
    
    return child1, child2

def mutate(chromosome, mutation_rate=0.05):
    """Randomly flip bits in the chromosome with a given probability."""
    for i in range(len(chromosome)):
        if random.random() < mutation_rate:
            chromosome[i] = 1 - chromosome[i] # Flips 1 to 0, or 0 to 1
    return chromosome

## Step 3: The Evolutionary Engine

Now we combine all the isolated mathematical components into a single loop representing the passage of time (generations). 

For each generation:
1. We evaluate the whole population.
2. We log the best individual for analytical tracking.
3. We generate a completely new population (the next generation) by repeatedly selecting parents, crossing them over, and mutating the offspring.

In [ ]:
# Hyperparameters for the Genetic Algorithm
POPULATION_SIZE = 100
GENERATIONS = 50
MUTATION_RATE = 0.05

def run_genetic_algorithm():
    # 1. Initialize the first generation
    population = initialize_population(POPULATION_SIZE, NUM_ITEMS)
    
    # Arrays to store analytical data for plotting
    history_best_fitness = []
    history_avg_fitness = []
    
    global_best_chromosome = None
    global_best_fitness = -1
    
    # 2. Evolution Loop
    for generation in range(GENERATIONS):
        # Calculate fitness for everyone
        fitness_scores = [calculate_fitness(ind) for ind in population]
        
        # Track metrics
        current_best_fitness = np.max(fitness_scores)
        current_avg_fitness = np.mean(fitness_scores)
        
        history_best_fitness.append(current_best_fitness)
        history_avg_fitness.append(current_avg_fitness)
        
        # Save the absolute best individual we've ever seen
        if current_best_fitness > global_best_fitness:
            global_best_fitness = current_best_fitness
            best_index = np.argmax(fitness_scores)
            global_best_chromosome = population[best_index]
        
        # 3. Create the next generation
        new_population = []
        # We need POPULATION_SIZE / 2 pairs of parents to create the new population
        for _ in range(POPULATION_SIZE // 2):
            parent1 = select_parent(population, fitness_scores)
            parent2 = select_parent(population, fitness_scores)
            
            child1, child2 = crossover(parent1, parent2)
            
            child1 = mutate(child1, MUTATION_RATE)
            child2 = mutate(child2, MUTATION_RATE)
            
            new_population.extend([child1, child2])
            
        # Replace old population with the new one
        population = new_population
        
    return global_best_chromosome, global_best_fitness, history_best_fitness, history_avg_fitness

# Execute the algorithm
best_solution, best_value, best_hist, avg_hist = run_genetic_algorithm()

print(f"Evolution Complete!")
print(f"Best Knapsack Value Achieved: {best_value}")
print(f"Optimal Item Selection (Chromosome): {best_solution}")
print(f"Total Weight of Optimal Selection: {np.sum(best_solution * weights)}")

## Step 4: Analytical Evaluation 

Machine Learning relies heavily on tracking metrics over time. By plotting the *Best Fitness* and *Average Fitness* per generation, we can visually diagnose the health of our Genetic Algorithm.

* If the line is flat immediately, your algorithm is trapped in a local optimum (increase mutation rate).
* If the average fitness bounces wildly and never stabilizes, your mutation rate might be too high, destroying good genetic patterns.

In [ ]:
# Visualize the learning curve
plt.figure(figsize=(10, 6))

plt.plot(best_hist, label='Best Fitness in Generation', color='green', linewidth=2)
plt.plot(avg_hist, label='Average Fitness in Generation', color='blue', linestyle='--')

plt.title('Genetic Algorithm Optimization over Generations', fontsize=16)
plt.xlabel('Generation (Time)', fontsize=12)
plt.ylabel('Fitness (Knapsack Value)', fontsize=12)

plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')
plt.show()

## Conclusion

You have successfully built an AI system that leverages the mathematical principles of natural selection! 

**Summary of what we learned:**
* **Fitness Functions** act as the environment, killing off bad solutions and rewarding good ones.
* **Crossover** exploits existing good solutions by combining their best traits.
* **Mutation** explores new areas of the mathematical search space to prevent stagnation.

While standard Machine Learning models (like Linear Regression) use calculus to find answers, Genetic Algorithms use probability and biology. They are invaluable tools for complex scheduling, routing (like the Traveling Salesperson Problem), and architecture search tasks.